# Klassed NEET Adaptive Engine: Authentic Training & Testing Pipeline

**Repository:** [udbhav968-creator/INETERNSHIP_WORK](https://github.com/udbhav968-creator/INETERNSHIP_WORK) (branch `fix/data-integrity-honest-metrics`)

This notebook provides a complete, self-contained, end-to-end training and evaluation pipeline for all 15 production machine learning and psychometric models across the 2,400 verified official NEET PYQ corpus.

### Pipeline Architecture
1. **3PL Item Response Theory (IRT)**: Item difficulty ($b$), discrimination ($a$), and pseudo-guessing ($c$) joint MAP estimation and 5-fold cross-validated cold-start Ridge regression.
2. **Empirical Bayes Bayesian Knowledge Tracing (BKT)**: 65 NCERT chapter prior distributions with hierarchical shrinkage.
3. **8-Skill DINA Cognitive Diagnosis**: Item-to-skill Q-matrix Expectation-Maximization calibration for slip ($s_j$) and guess ($g_j$) parameters.
4. **Truncated SVD / TF-IDF LSA RAG Engine**: 32-dimensional semantic knowledge indexing across NCERT physics, chemistry, and biology.
5. **Random Forest 300-Tree Difficulty Classifier**: 5-fold Stratified Cross-Validation, hyperparameter grid search, permutation importance, and compact NumPy serverless runtime export.
6. **Temperature Scaling Calibration**: Post-hoc Expected Calibration Error (ECE) optimization via L-BFGS-B.
7. **Conformal Prediction 180-Q Monte Carlo Simulator**: Distribution-free coverage intervals and negative marking optimal threshold policy.
8. **Full MLOps Validation Suite**: 25 standalone pure-ML testing suites, 29 backend integration suites, and SHA-256 artifact manifest verification.

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q scikit-learn numpy pandas scipy joblib fastapi uvicorn pydantic pydantic[email] bcrypt cryptography matplotlib seaborn tabulate
import os, sys
if not os.path.exists('model') and not os.path.exists('data'):
    !git clone -b fix/data-integrity-honest-metrics https://github.com/udbhav968-creator/INETERNSHIP_WORK.git repo
    os.chdir('repo')
sys.path.insert(0, os.getcwd())
print('Active working directory:', os.getcwd())


## 2. Dataset Verification & Data Integrity Gate

In [ ]:
import json, pandas as pd
from model.data_integrity_gate import run_data_integrity_gate
gate_report = run_data_integrity_gate()
print('Data Integrity Gate Status:', gate_report.get('status'))
print('Total Verified Questions:', gate_report.get('verified_questions_count'))
print('Quarantined Questions:', gate_report.get('quarantined_questions_count'))
print('Chi-Square Uniformity p-value:', gate_report.get('chi_square_uniformity_test', {}).get('p_value'))
df_features = pd.read_csv('data/training/question_features.csv')
print('Loaded feature rows:', len(df_features))
df_features[['subject', 'difficulty_level', 'ideal_time_sec', 'word_count']].describe()


## 3. Phase 1: 3PL Item Response Theory & Cold-Start Ridge Regressor Training

Estimates item discrimination ($a_i$), difficulty ($b_i$), and pseudo-guessing ($c_i$) parameters using empirical attempt statistics and 3PL logistic formulation:

$$P(Y_{ij} = 1 \mid \theta_j) = c_i + (1 - c_i) \frac{1}{1 + e^{-1.702 a_i (\theta_j - b_i)}}$$

Followed by 5-fold cross-validated Ridge regression with grid search over $\alpha \in [0.1, 10.0]$.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from model.irt_3pl_model import calibrate_question_bank, fisher_information, irt_3pl_probability, IRTItemParams
from model.train_psychometric_and_rag_models import train_irt_cold_start_ridge_regressor
calibrated_items = calibrate_question_bank()
print('Calibrated Question Count:', len(calibrated_items))
ridge_metrics = train_irt_cold_start_ridge_regressor(df_features, calibrated_items)
print('IRT Ridge Regressor 5-Fold Results:')
print(json.dumps(ridge_metrics, indent=2))
thetas = np.linspace(-3.0, 3.0, 100)
plt.figure(figsize=(12, 4))
for diff, p_item in [('basic', IRTItemParams(1.05, -1.15, 0.25)), ('medium', IRTItemParams(1.32, 0.05, 0.22)), ('advanced', IRTItemParams(1.68, 1.30, 0.18))]:
    probs = [irt_3pl_probability(th, p_item) for th in thetas]
    infos = [fisher_information(th, p_item) for th in thetas]
    plt.subplot(1, 2, 1); plt.plot(thetas, probs, label=f'{diff} (b={p_item.b})')
    plt.subplot(1, 2, 2); plt.plot(thetas, infos, label=f'{diff}')
plt.subplot(1, 2, 1); plt.title('3PL Item Characteristic Curves (ICC)'); plt.xlabel('Latent Ability (theta)'); plt.ylabel('P(Correct)'); plt.legend(); plt.grid(True)
plt.subplot(1, 2, 2); plt.title('Fisher Information Curves'); plt.xlabel('Latent Ability (theta)'); plt.ylabel('Information I(theta)'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


## 4. Phase 2: Bayesian Knowledge Tracing (BKT) 65-Chapter Priors & State Transitions

Calibrates Empirical Bayes mastery priors $p(L_0)$ across all 65 NCERT canonical chapters based on NEET historical weightage and hierarchical DAG prerequisite shrinkage.

In [ ]:
from model.train_psychometric_and_rag_models import train_chapter_wise_bkt_parameters
with open('data/tagged_questions/neet_questions_clean.json', encoding='utf-8') as f:
    clean_qs = json.load(f)
with open('model/topic_importance.json', encoding='utf-8') as f:
    topic_imp = json.load(f)
bkt_metrics = train_chapter_wise_bkt_parameters(clean_qs, calibrated_items, topic_imp)
print('BKT Chapter Empirical Bayes Training Results:')
print(f"Total Chapters: {bkt_metrics['total_chapters']}")
print(f"Mastery Prior Mean: {bkt_metrics['mastery_prior_mean']:.4f}")
print(f"Transit Prior Mean: {bkt_metrics['transit_prior_mean']:.4f}")
with open('model/bkt_chapter_parameters.json', encoding='utf-8') as f:
    bkt_registry = json.load(f)
top_chapters = sorted(bkt_registry.items(), key=lambda kv: kv[1]['p_l0'], reverse=True)
print('Top 3 Highest Prior Mastery Chapters:')
for ch, p in top_chapters[:3]:
    print(f"  {ch}: p_l0={p['p_l0']:.3f}, transit={p['p_transit']:.3f}")


## 5. Phase 3: 8-Skill DINA Cognitive Diagnosis Model

Estimates deterministic input, noisy "and" gate (DINA) psychometric parameters for 8 fine-grained cognitive skills. Estimates slip ($s_j$) and guess ($g_j$) rates for each skill dimension.

In [ ]:
from model.train_psychometric_and_rag_models import train_dina_8skill_q_matrix_parameters
dina_metrics = train_dina_8skill_q_matrix_parameters(clean_qs, calibrated_items)
print('DINA 8-Skill Cognitive Diagnosis Metrics:')
for skill_id, params in dina_metrics['skills'].items():
    print(f"{skill_id:32} | Items: {params['tagged_items_count']:4d} | Prior: {params['prior_mastery']:.3f} | Slip: {params['slip_rate']:.3f} | Guess: {params['guess_rate']:.3f}")


## 6. Phase 4: Hybrid TF-IDF & 32-Dimensional Truncated SVD Socratic RAG Index

Constructs a Latent Semantic Analysis (LSA) vector space index using sublinear term-frequency scaling and 32-dimensional SVD decomposition across all 65 NCERT knowledge base chapters.

In [ ]:
from model.train_psychometric_and_rag_models import train_ncert_and_pyq_svd_rag_index
svd_metrics = train_ncert_and_pyq_svd_rag_index(clean_qs)
print('NCERT SVD/LSA RAG Index Metrics:')
print(f"Chapters Indexed: {svd_metrics['n_chapters_indexed']}")
print(f"Vocabulary Size: {svd_metrics['vocab_size']}")
print(f"LSA Dimensions: {svd_metrics['lsa_dimensions']}")
print(f"Explained Variance: {svd_metrics['lsa_explained_variance_ratio']:.4f}")
from model.rag_socratic_engine import SocraticRAGIndex
rag_idx = SocraticRAGIndex()
test_query = 'Derive the magnetic field inside a long current carrying solenoid using Ampere circuital law'
results = rag_idx.query_hybrid(test_query, top_k=3)
print('\nRAG Test Query:', test_query)
for r in results:
    print(f"Chapter: {r['chapter']} | Similarity: {r['similarity']:.4f}")


## 7. Phase 5: Random Forest 300-Tree Classifier & Tournament Benchmark

Executes 5-fold Stratified Cross-Validation on the 2,400 questions, grid-search hyperparameter fine-tuning, permutation importance, and champion-challenger tournament against ExtraTrees and Logistic Regression.

In [ ]:
from model.train_difficulty_model import main as train_rf_main
from model.training_and_validation_pipeline import run_deep_training_and_validation
train_rf_main()
full_report = run_deep_training_and_validation(save_artifacts=True)
cv_results = full_report['stratified_5fold_cross_validation']
print('5-Fold Cross Validation Accuracy:', cv_results['mean_cv_accuracy'], '+/-', cv_results['std_cv_accuracy'])
print('5-Fold Macro F1:', cv_results['mean_cv_macro_f1'])
print('Out-of-Fold Brier Score:', cv_results['oof_multiclass_brier_score'])
print('\nChampion-Challenger Tournament Comparison:')
for model_name, res in full_report['champion_challenger_tournament'].items():
    acc = res.get('holdout_accuracy', res.get('accuracy', 0.0))
    print(f"{model_name:25} | Accuracy: {acc:.4f} | Macro F1: {res['macro_f1']:.4f}")


## 8. Phase 6: Temperature Scaling & Probability Calibration Optimization

Fits temperature parameter $T > 0$ to calibrate softmax prediction probabilities:

$$\hat{p}_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Optimizes Negative Log-Likelihood (NLL) and minimizes 10-bin Expected Calibration Error (ECE).

In [ ]:
from model.probability_calibrator import fit_and_save_rf_temperature_calibration, get_calibration_metadata
temp_results = fit_and_save_rf_temperature_calibration()
print('Optimal Temperature T:', temp_results['optimal_temperature'])
print('Pre-Calibration ECE (10-bin):', temp_results['pre_calibration']['ece_10_bin'])
print('Post-Calibration ECE (10-bin):', temp_results['post_calibration']['ece_10_bin'])
print('Pre-Calibration NLL:', temp_results['pre_calibration']['multiclass_nll'])
print('Post-Calibration NLL:', temp_results['post_calibration']['multiclass_nll'])


## 9. Phase 7: Conformal Prediction 180-Question Monte Carlo Simulation

Runs 10,000 trials of full 180-question NEET exam simulations under distribution-free conformal calibration guarantees ($90\%$ prediction interval) and solves for the optimal negative marking risk-reward threshold.

In [ ]:
from model.monte_carlo_conformal_neet import run_monte_carlo_conformal_neet_simulation
sim_results = run_monte_carlo_conformal_neet_simulation(
    student_id='COLAB_BENCHMARK',
    global_theta=0.5,
    mastery_map={ch: 0.55 for ch in topic_imp},
    theta_map={ch: 0.5 for ch in topic_imp},
    ci_map={ch: 0.35 for ch in topic_imp},
    topic_importance=topic_imp,
)
print('Monte Carlo Trials:', sim_results['n_trials'])
print('Expected 180Q Score:', sim_results.get('expected_score_attempt_all_720', sim_results.get('mean_expected_score_720', 0.0)))
print('Conformal 90% Confidence Interval:', sim_results['conformal_90_interval'])
print('Interquartile 50% Range:', sim_results['interquartile_50_interval'])
opt = sim_results['negative_marking_optimizer']
print('Negative Marking Optimal Threshold:', opt['optimal_skip_threshold_prob'])
print('Optimal Skip Expected Score:', opt['optimal_skip_expected_score_720'])
print('Expected Marks Gained by Skipping P < threshold:', opt['expected_marks_gained_by_skipping'])


## 10. Phase 8: Standalone ML Test Suites & Backend Integration Suites

Executes all 25 pure-ML & MLOps standalone suites and all 29 backend integration suites, followed by SHA-256 artifact integrity manifest verification.

In [ ]:
!python -m model.tests.test_ml_standalone_suite
!python backend/tests/test_v2_engine.py
from model.verify_artifact_integrity import verify_all_ml_artifacts
integ = verify_all_ml_artifacts(allow_serverless_omissions=False, require_clean_bandit=True)
print('\nArtifact Integrity Verification Results:')
print('Integrity Status:', integ['integrity_status'])
print('Verified Artifacts Count:', integ['verified_artifacts_count'])
print('Calibrated IRT Items:', integ['calibrated_irt_items'])
print('Trained BKT Chapters:', integ['trained_bkt_chapters'])
print('Trained DINA Skills:', integ['trained_dina_skills'])
